# 05b · GSE65391 · RNA_array · scoring every sample on the first-visit modules

Reads `modules.rds`, `wgcna_input.rds`, `expression.rds`. Writes `projected_scores.rds` and the end
product `module_loadings.csv` in `data/run_artifacts/GSE65391/`.

**Why this step.** The modules and their eigengenes were fitted on 157 samples, the first visit of each
child (notebooks 04 and 05). The other 767 SLE samples (later visits, and the one sample with no visit
number) and the 48 healthy samples have no eigengene, because they were not in the fit. Refitting on all
samples would change each module and give children with many visits more weight. Instead, every sample is
scored with the fitted model. This is the standard way to place new samples on principal components learned
from a training set (Jolliffe IT. *Principal Component Analysis*, 2nd ed., Springer 2002, sections 3.1 and
9.2; Sharma G et al. projectR. *Bioinformatics* 2020;36:3592-3593).

**The model.** A WGCNA module eigengene is the first principal component of the module's genes, each gene
standardised across the training samples (Langfelder P, Horvath S. Eigengene networks for studying the
relationships between co-expression modules. *BMC Syst Biol* 2007;1:54). WGCNA computes it with a singular
value decomposition (SVD) of the standardised genes by samples matrix Z: Z = U D V', where ' means
transposed. The eigengene is the first column of V. Because that column equals t(Z) times u1 divided by d1
(u1 the first column of U, d1 the first value of D), a new sample is scored as

  score = sum over the module's genes of z times weight, where z = (x - mean) / SD and weight = u1 / d1.

The mean and SD of each gene are those of the 157 first visits. The model for each module is therefore: the
gene list, each gene's first-visit mean and SD, and the weights.

**Round-trip gate.** Scoring the 157 first visits with the model must reproduce the WGCNA eigengenes of
notebook 05: Pearson r of 1 and a largest absolute difference at the level of rounding error. Nothing is
scored until the gate passes.

## Load the modules, the first-visit matrix and all samples

In [1]:
source("../src/paths.R")
mods <- readRDS(art("GSE65391", "modules.rds"))
w    <- readRDS(art("GSE65391", "wgcna_input.rds"))
x    <- readRDS(art("GSE65391", "expression.rds"))
fv   <- rownames(w$datExpr)
modules <- setdiff(unique(mods$colors), "grey")
c(first_visits = length(fv), modules = length(modules), all_samples = ncol(x$E))

first_visits      modules  all_samples 
         157           14          972

**Explanation**
- The three files come from notebooks 05, 04 and 03b (see notebook 00 for `art()`).
- `fv` holds the identifiers of the 157 first-visit samples: the row names of the WGCNA matrix.
- `unique(mods$colors)` lists the module colours once each; `setdiff(..., "grey")` removes grey, which is
  not a module. For example, `setdiff(c("pink", "grey", "tan"), "grey")` gives `"pink" "tan"`.

## Fit the model for each module

In [2]:
model <- lapply(setNames(modules, modules), function(mo) {
  g  <- names(mods$colors)[mods$colors == mo]
  X  <- w$datExpr[, g]                                   # first visits x module genes
  mu <- colMeans(X); s <- apply(X, 2, sd)
  Z  <- t((t(X) - mu) / s)                                # standardised, samples x genes
  sv <- svd(t(Z), nu = 1, nv = 1)                          # genes x samples = U D V'
  wt <- sv$u[, 1] / sv$d[1]
  sc <- Z %*% wt                                           # equals sv$v[, 1]
  if (cor(sc, rowMeans(Z)) < 0) wt <- -wt                  # WGCNA's alignment with average expression
  data.frame(module = mo, gene = g, mean = mu, sd = s, weight = wt, row.names = NULL)
})

**Explanation**
- `lapply(setNames(modules, modules), function(mo) ...)` runs the function once for each module and names
  the results by module (see notebook 05 for `setNames`).
- `g` holds the module's genes; `X` holds their values in the 157 first visits.
- `colMeans(X)` gives each gene's mean; `apply(X, 2, sd)` gives each gene's standard deviation. The `2`
  means "for each column".
- `Z` holds the standardised values: each gene minus its mean, divided by its SD. `t()` transposes twice so
  that the subtraction runs gene by gene.
- `svd(t(Z), nu = 1, nv = 1)` computes the decomposition of the genes by samples matrix and keeps only the
  first column of U and of V.
- `wt` holds the weights, u1 / d1.
- `Z %*% wt` is matrix multiplication: for each sample, the sum over genes of z times weight. This is the
  score, and it equals the first column of V.
- A principal component can point either way. WGCNA chooses the direction in which the eigengene rises with
  the module's average expression. `cor(sc, rowMeans(Z)) < 0` tests for the other direction, and `-wt`
  reverses the weights if so.
- The function returns one row per gene: module, gene, mean, SD and weight.

## Round-trip gate

In [3]:
score <- function(E, mdl) {                               # E: genes x samples
  sapply(mdl, function(m) colSums(((E[m$gene, , drop = FALSE] - m$mean) / m$sd) * m$weight))
}
fv_scores <- score(t(w$datExpr), model)
gate <- data.frame(module = modules,
                   r = sapply(modules, function(mo) cor(fv_scores[, mo], mods$MEs[fv, paste0("ME", mo)])),
                   max_abs_diff = sapply(modules, function(mo) max(abs(fv_scores[, mo] - mods$MEs[fv, paste0("ME", mo)]))),
                   row.names = NULL)
gate
stopifnot(all(gate$r > 1 - 1e-10), all(gate$max_abs_diff < 1e-8))
cat("GATE PASSED: the model reproduces the WGCNA eigengenes of the 157 first visits\n")

module,r,max_abs_diff
<chr>,<dbl>,<dbl>
turquoise,1,1.630640e-15
cyan,1,1.998401e-15
blue,1,9.159340e-16
red,1,8.187895e-16
salmon,1,2.803313e-15
brown,1,1.513546e-15
yellow,1,1.318390e-15
tan,1,3.608225e-16
black,1,3.330669e-15


GATE PASSED: the model reproduces the WGCNA eigengenes of the 157 first visits


**Explanation**
- `score(E, mdl)` takes a genes by samples matrix and the model, and returns a samples by modules matrix of
  scores.
  - For each module `m`: `E[m$gene, ]` takes its genes; `(... - m$mean) / m$sd` standardises them with the
    first-visit mean and SD; `* m$weight` multiplies by the weights; `colSums()` adds up over genes, giving
    one score per sample.
  - `sapply()` does this for every module and puts the results side by side.
- `fv_scores` scores the 157 first visits.
- `gate` compares them with the WGCNA eigengenes of notebook 05, module by module: `r` is the Pearson
  correlation, `max_abs_diff` the largest absolute difference.
- `stopifnot()` stops the notebook with an error if any condition is FALSE. Here: every r above
  1 - 1e-10 and every difference below 1e-8 (0.00000001).
- `cat()` prints the message only if the gate passed.

**Result.** Gate passed for all 14 modules: r = 1, largest absolute difference 3e-15
(0.000000000000003).

## Score every array sample

In [4]:
stopifnot(all(unlist(lapply(model, `[[`, "gene")) %in% rownames(x$E)))
all_scores <- score(x$E, model)
m <- x$meta[rownames(all_scores), ]
table(disease = m$disease, first_visit = m$first_visit)

         first_visit
disease   FALSE TRUE
  Healthy     2   46
  SLE       767  157

**Explanation**
- The first line checks that every module gene is in the full matrix. `lapply(model, `[[`, "gene")` takes
  the `gene` column of each module; `unlist()` joins them into one list.
- `score(x$E, model)` scores all 972 array samples: 924 SLE samples (every visit) and 48 healthy samples.
- `m` holds the sample table in the same order as the scores.
- The table counts samples by disease and first visit.
- `first_visit` is also TRUE for each healthy child's first sample (notebook 01). Only the SLE first visits
  were used to fit the modules; the healthy samples are scored, not fitted.

**Result.** All 972 array samples are scored: 157 SLE first visits, 767 other SLE samples and 48
healthy samples.

## Save

In [5]:
saveRDS(list(scores = all_scores, meta = m, model = model, gate = gate), art("GSE65391", "projected_scores.rds"))
write.csv(do.call(rbind, model), art("GSE65391", "module_loadings.csv"), row.names = FALSE)

**Explanation**
- `projected_scores.rds` keeps the scores, the sample table, the model and the gate for later notebooks.
- `do.call(rbind, model)` stacks the 14 module tables into one table.
- `module_loadings.csv` is the end product the other studies receive: module, gene, first-visit mean and SD,
  and weight. It contains no expression values of any sample.